# PenuX SAST Pipeline
מערכת אוטומטית לפענוח (Decompilation), עיבוד (Unminify) וסריקת חולשות (SAST) בקובצי APK.

In [ ]:
# התקנת תלויות מערכת (Apktool) וספריות Python
!apt-get update -qq
!apt-get install -y -qq apktool
!pip install -q jsbeautifier pandas

print("[+] ההתקנות הושלמו בהצלחה.")

In [ ]:
import os
import re
import subprocess
import pandas as pd
import jsbeautifier
from google.colab import drive
from IPython.display import display, HTML

# חיבור ל-Drive (ידרוש אישור בהרצה הראשונה)
drive.mount('/content/drive')

# --- הגדרות נתיבים ---
APK_TARGET_PATH = '/content/drive/MyDrive/TargetApp.apk'
DECOMPILED_DIR = '/content/extracted_apk'

In [ ]:
def decompile_apk(apk_path, output_dir):
    print(f"[*] מתחיל בפענוח ה-APK: {apk_path}")
    if not os.path.exists(apk_path):
        print(f"[!] שגיאה: קובץ לא נמצא בנתיב {apk_path}")
        return False
        
    command = ["apktool", "d", "-f", apk_path, "-o", output_dir]
    try:
        subprocess.run(command, capture_output=True, text=True, check=True)
        print(f"[+] ה-APK פוענח בהצלחה לתיקייה: {output_dir}")
        return True
    except subprocess.CalledProcessError as e:
        print(f"[!] שגיאת פריסה:\n{e.stderr}")
        return False

def unminify_js_files(directory):
    print("[*] מחפש קובצי JavaScript דחוסים לביצוע Unminify...")
    opts = jsbeautifier.default_options()
    processed_count = 0
    
    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith('.bundle') or file.endswith('.js'):
                filepath = os.path.join(root, file)
                try:
                    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                        code = f.read()
                    
                    # זיהוי קובץ מוקטן: מעט שורות אך משקל משמעותי
                    if code.count('\n') < 20 and len(code) > 5000:
                        print(f"    -> מבצע Unminify לקובץ: {file}")
                        formatted_code = jsbeautifier.beautify(code, opts)
                        with open(filepath, 'w', encoding='utf-8') as f:
                            f.write(formatted_code)
                        processed_count += 1
                except Exception as e:
                    print(f"    [!] שגיאה בעיבוד {file}: {e}")
                    
    print(f"[+] הושלם עיצוב מחדש עבור {processed_count} קבצים.")

In [ ]:
RULES = [
    {
        "id": "RCE_WORKLET_INJECTION",
        "category": "Code Injection",
        "pattern": r"(eval|new\s+Function|setTimeout|setInterval)\s*\(\s*[^)]*(?:_worklet_|reanimated).*?\)",
        "severity": "CRITICAL",
        "description": "הרצת קוד דינמי ולא מאובטח בסביבת React Native Reanimated."
    },
    {
        "id": "EXPOSED_JWT_TOKEN",
        "category": "Secrets",
        "pattern": r"ey[A-Za-z0-9-_=]+\.[A-Za-z0-9-_=]+\.?[A-Za-z0-9-_.+/=]*",
        "severity": "CRITICAL",
        "description": "זיהוי מבנה של אסימון JWT חשוף."
    },
    {
        "id": "EXPOSED_AWS_KEY",
        "category": "Secrets",
        "pattern": r"(A3T[A-Z0-9]|AKIA|AGPA|AIDA|AROA|AIPA|ANPA|ANVA|ASIA)[A-Z0-9]{16}",
        "severity": "CRITICAL",
        "description": "זיהוי מפתח גישה פוטנציאלי של AWS."
    },
    {
        "id": "HARDCODED_B2C_ENDPOINT",
        "category": "Authentication",
        "pattern": r"companyb2cprod\.onmicrosoft\.com",
        "severity": "HIGH",
        "description": "חשיפת נקודת קצה של סביבת ייצור (Hardcoded)."
    },
    {
        "id": "INSECURE_LOCAL_STORAGE",
        "category": "Data Storage",
        "pattern": r"AsyncStorage\.setItem\s*\(\s*['\"][^'\"]*(token|password|secret|auth)[^'\"]*['\"]",
        "severity": "HIGH",
        "description": "שמירת מידע רגיש באחסון מקומי ללא הצפנה."
    },
    {
        "id": "OVERPRIVILEGED_MANIFEST",
        "category": "Permissions",
        "pattern": r"<uses-permission\s+android:name=\"android\.permission\.(ACCESS_BACKGROUND_LOCATION|READ_PHONE_STATE|BODY_SENSORS_BACKGROUND)\"\s*/>",
        "severity": "MEDIUM",
        "description": "דרישת הרשאות מופרזות במניפסט."
    }
]

def scan_directory(directory_path):
    findings = []
    for root, _, files in os.walk(directory_path):
        for file_name in files:
            if not re.search(r'\.(js|ts|json|xml|java|kt)$|bundle', file_name, re.IGNORECASE):
                continue
                
            file_path = os.path.join(root, file_name)
            try:
                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read()
                    lines = content.split('\n')
                    
                for rule in RULES:
                    for match in re.finditer(rule["pattern"], content, re.IGNORECASE):
                        start_idx = match.start()
                        line_num = content.count('\n', 0, start_idx) + 1
                        
                        snippet = lines[line_num - 1].strip()
                        if len(snippet) > 150:
                            snippet = snippet[:150] + "..."
                            
                        findings.append({
                            "Severity": rule["severity"],
                            "Category": rule["category"],
                            "Rule ID": rule["id"],
                            "File": file_name,
                            "Line": line_num,
                            "Description": rule["description"],
                            "Snippet": snippet
                        })
            except Exception as e:
                pass
                
    return findings

In [ ]:
print("=" * 60)
print("🚀 מתחיל תהליך פריסה וסריקת SAST...")
print("=" * 60)

if decompile_apk(APK_TARGET_PATH, DECOMPILED_DIR):
    unminify_js_files(DECOMPILED_DIR)
    
    print(f"[*] מריץ סורק חתימות על סביבת הנייטיב וה-JS...")
    raw_findings = scan_directory(DECOMPILED_DIR)

    if raw_findings:
        df = pd.DataFrame(raw_findings)
        
        severity_order = {"CRITICAL": 1, "HIGH": 2, "MEDIUM": 3, "LOW": 4}
        df['Severity_Rank'] = df['Severity'].map(severity_order)
        df = df.sort_values(by=['Severity_Rank', 'Category']).drop('Severity_Rank', axis=1)
        
        print(f"\n[+] הסריקה הסתיימה בהצלחה! נמצאו {len(df)} פגיעויות אפשריות.")
        display(HTML(df.to_html(index=False, classes='table table-striped')))
        
        output_csv = '/content/drive/MyDrive/sast_pipeline_report.csv'
        df.to_csv(output_csv, index=False, encoding='utf-8-sig')
        print(f"\n[*] דוח ה-CSV הופק ונשמר בהצלחה בנתיב: {output_csv}")
        
    else:
        print("\n[+] הסריקה הסתיימה בהצלחה - לא נמצאו פגיעויות תואמות למילון החוקים.")
else:
    print("\n[!] התהליך נעצר עקב שגיאה בפענוח ה-APK. אנא ודא שהנתיב תקין.")